In [ ]:
import pandas as pd
import argparse
from pathlib import Path


# --------------------------------------------------
# Kendte skoledomæner
# --------------------------------------------------
SCHOOL_DOMAINS = {
    "sde.dk": "Syddansk Erhvervsskole",
    "tec.dk": "TEC",
    "techcollege.dk": "TECHCOLLEGE",
    "celf.dk": "CELF",
    "eucsj.dk": "EUC Sjælland",
    "aarhustech.dk": "Aarhus Tech",
    "herningsholm.dk": "Herningsholm Erhvervsskole & Gymnasier",
    "rts.dk": "Roskilde Tekniske Skole",
}


# --------------------------------------------------
# Normalisering af skolenavne
# --------------------------------------------------
SCHOOL_NAME_MAP = {
    "sde": "Syddansk Erhvervsskole",
    "syddansk erhvervsskole": "Syddansk Erhvervsskole",

    "tec": "TEC",

    "techcollege": "TECHCOLLEGE",
    "tech college": "TECHCOLLEGE",

    "celf": "CELF",

    "euc sjælland": "EUC Sjælland",
    "euc sjaelland": "EUC Sjælland",

    "aarhus tech": "Aarhus Tech",
    "århus tech": "Aarhus Tech",

    "herningsholm": "Herningsholm Erhvervsskole & Gymnasier",
    "herningsholm erhvervsskole": "Herningsholm Erhvervsskole & Gymnasier",

    "roskilde tekniske skole": "Roskilde Tekniske Skole",
    "rts": "Roskilde Tekniske Skole",
}


# --------------------------------------------------
# Tegnkodning
# --------------------------------------------------
def fix_mojibake(value):
    """
    Forsøger fx at rette:
    JÃ¸rgen -> Jørgen
    SÃ¸ren  -> Søren
    SjÃ¦lland -> Sjælland
    """

    if pd.isna(value):
        return ""

    text = str(value)

    try:
        if any(x in text for x in ["Ã", "Â", "â"]):
            return text.encode("latin1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass

    return text


# --------------------------------------------------
# Maildomæne
# --------------------------------------------------
def get_email_domain(email):

    if pd.isna(email):
        return ""

    email = str(email).strip().lower()

    if "@" not in email:
        return ""

    return email.split("@")[-1]


# --------------------------------------------------
# Skolenavn
# --------------------------------------------------
def normalize_school_name(value):

    if pd.isna(value):
        return ""

    value = fix_mojibake(value).strip()

    if not value:
        return ""

    key = value.lower()

    if key in SCHOOL_NAME_MAP:
        return SCHOOL_NAME_MAP[key]

    return value


# --------------------------------------------------
# Klassificering
# --------------------------------------------------
def clean_row(
    row,
    email_col,
    school_col,
    company_col
):

    changes = []

    email = fix_mojibake(row.get(email_col, ""))
    school = fix_mojibake(row.get(school_col, ""))
    company = fix_mojibake(row.get(company_col, ""))

    domain = get_email_domain(email)

    original_school = school
    original_company = company

    # ----------------------------------------------
    # Kendt skoledomæne vinder
    # ----------------------------------------------
    if domain in SCHOOL_DOMAINS:

        detected_school = SCHOOL_DOMAINS[domain]

        if school != detected_school:

            changes.append(
                f"Skole ændret fra '{school}' "
                f"til '{detected_school}' ud fra {domain}"
            )

        school = detected_school

        sender_type = "Skole/institution (fra emaildomæne)"

    else:

        school = normalize_school_name(school)

        sender_type = "Virksomhed/organisation/ukendt"

    # ----------------------------------------------
    # Log ændringer
    # ----------------------------------------------
    if original_school != school and not changes:

        changes.append(
            f"Skolenavn normaliseret: "
            f"'{original_school}' -> '{school}'"
        )

    if original_company != company:

        changes.append(
            f"Virksomhed ændret: "
            f"'{original_company}' -> '{company}'"
        )

    return pd.Series({
        "email_domain": domain,
        "subscriber_virksomhed_skole": school,
        "subscriber_virksomhed": company,
        "afsender_type": sender_type,
        "ændringslog": " | ".join(changes)
    })


# --------------------------------------------------
# Rens hele datasættet
# --------------------------------------------------
def clean_subscribers(
    input_file,
    output_file=None,
    email_col="Email",
    name_col="Navn",
    school_col="subscriber_virksomhed_skole",
    company_col="subscriber_virksomhed"
):

    print(f"Læser: {input_file}")

    df = pd.read_excel(
        input_file,
        engine="openpyxl"
    )

    # Ret kodning generelt
    for col in df.select_dtypes(include="object").columns:

        df[col] = df[col].apply(fix_mojibake)

    # Kontrollér nødvendige kolonner
    required = [
        email_col,
        school_col,
        company_col
    ]

    missing = [
        col for col in required
        if col not in df.columns
    ]

    if missing:

        raise ValueError(
            f"Manglende kolonner: {missing}"
        )

    # Rens
    cleaned = df.apply(
        lambda row: clean_row(
            row,
            email_col,
            school_col,
            company_col
        ),
        axis=1
    )

    # Overskriv/tilføj felter
    for col in cleaned.columns:

        df[col] = cleaned[col]

    # ----------------------------------------------
    # Domæneoversigt
    # ----------------------------------------------
    domain_summary = (
        df.groupby(
            [
                "email_domain",
                "afsender_type",
                "subscriber_virksomhed_skole"
            ],
            dropna=False
        )
        .size()
        .reset_index(name="antal")
        .sort_values(
            "antal",
            ascending=False
        )
    )

    # ----------------------------------------------
    # Rækker til manuel gennemgang
    # ----------------------------------------------
    review = df[
        df["afsender_type"]
        == "Virksomhed/organisation/ukendt"
    ].copy()

    # ----------------------------------------------
    # Outputfil
    # ----------------------------------------------
    if output_file is None:

        path = Path(input_file)

        output_file = (
            path.parent /
            f"{path.stem}_renset.xlsx"
        )

    with pd.ExcelWriter(
        output_file,
        engine="openpyxl"
    ) as writer:

        df.to_excel(
            writer,
            sheet_name="Renset_data",
            index=False
        )

        domain_summary.to_excel(
            writer,
            sheet_name="Domaene_oversigt",
            index=False
        )

        review.to_excel(
            writer,
            sheet_name="Til_gennemgang",
            index=False
        )

    print()
    print("--------------------------------")
    print("Rensning færdig")
    print("--------------------------------")
    print(f"Antal rækker: {len(df)}")
    print(
        "Skoler:",
        (
            df["afsender_type"]
            == "Skole/institution (fra emaildomæne)"
        ).sum()
    )
    print(
        "Virksomhed/ukendt:",
        (
            df["afsender_type"]
            == "Virksomhed/organisation/ukendt"
        ).sum()
    )
    print(f"Output: {output_file}")


# --------------------------------------------------
# Kør fra terminal
# --------------------------------------------------
if __name__ == "__main__":

    parser = argparse.ArgumentParser(
        description="Rens subscribers til Vidensportalen"
    )

    parser.add_argument(
        "input_file"
    )

    parser.add_argument(
        "--output"
    )

    parser.add_argument(
        "--email-col",
        default="Email"
    )

    parser.add_argument(
        "--name-col",
        default="Navn"
    )

    parser.add_argument(
        "--school-col",
        default="subscriber_virksomhed_skole"
    )

    parser.add_argument(
        "--company-col",
        default="subscriber_virksomhed"
    )

    args = parser.parse_args()

    clean_subscribers(
        input_file=args.input_file,
        output_file=args.output,
        email_col=args.email_col,
        name_col=args.name_col,
        school_col=args.school_col,
        company_col=args.company_col
    )